# Mathematical and Computational Background

**IMSE 774 · Neural Networks · Week 2 · Sessions 3–4 · September 1 & 3, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/02-math-background.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"

## Vectors, Matrices, and Tensors


`shapes`


In [ ]:
x = np.array([1.0, 2.0, 3.0])                  # vector in R^3
Y = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])                # matrix in R^{2x3}
Z = np.arange(24, dtype=float).reshape(2, 3, 4)  # 3D tensor

for name, obj in [("x", x), ("Y", Y), ("Z", Z)]:
    print(f"{name}: shape {obj.shape}, ndim {obj.ndim}, {obj.size} numbers")

# PyTorch calls all three "tensors" -- only the shape distinguishes them.
xt, Yt, Zt = torch.tensor(x), torch.tensor(Y), torch.tensor(Z)
print("\ntorch types:", type(xt).__name__, type(Yt).__name__, type(Zt).__name__)
print("torch shapes:", tuple(xt.shape), tuple(Yt.shape), tuple(Zt.shape))

`column-vs-row`


In [ ]:
v = np.array([1.0, 2.0, 3.0])
print("1-D array      :", v.shape, "-- neither row nor column")
print("explicit column:", v.reshape(-1, 1).shape)
print("explicit row   :", v.reshape(1, -1).shape)

# The difference is not cosmetic: broadcasting treats these completely differently.
print("\ncolumn - row gives an outer difference:")
print(v.reshape(-1, 1) - v.reshape(1, -1))

## Transpose


`transpose`


In [ ]:
A = np.random.randn(3, 4)
B = np.random.randn(4, 2)

print("A:", A.shape, " B:", B.shape, " AB:", (A @ B).shape, " (AB)^T:", (A @ B).T.shape)
print("(AB)^T == B^T A^T :", np.allclose((A @ B).T, B.T @ A.T))
print("A^T B^T is not even defined:", A.T.shape, "x", B.T.shape)

## Vector and matrix norms


`norms`


In [ ]:
z = np.array([3.0, -4.0, 12.0])
for p in [1, 2, np.inf]:
    print(f"||z||_{p!s:<4} = {np.linalg.norm(z, p):.4f}")

Z = np.array([[1.0, 2.0], [3.0, 4.0]])
print("\n||Z||_F              =", round(np.linalg.norm(Z, "fro"), 4))
print("Euclidean norm of the flattened matrix =", round(np.linalg.norm(Z.ravel()), 4))

`fig-norm-balls`


In [ ]:
theta = np.linspace(0, 2 * np.pi, 721)
dirs = np.stack([np.cos(theta), np.sin(theta)])       # unit directions, shape (2, 721)

def unit_ball(p):
    """Rescale each direction so that its p-norm is exactly one."""
    if np.isinf(p):
        scale = np.max(np.abs(dirs), axis=0)
    else:
        scale = np.sum(np.abs(dirs) ** p, axis=0) ** (1 / p)
    return dirs / scale

fig, ax = plt.subplots(figsize=(4.6, 4.6))
for p, label, color in [(1, r"$p = 1$", ACCENT),
                        (2, r"$p = 2$", NDSU_GREEN),
                        (np.inf, r"$p = \infty$", "#4C6EF5")]:
    ball = unit_ball(p)
    ax.plot(ball[0], ball[1], color=color, lw=2, label=label)

ax.set_aspect("equal")
ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45)
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
ax.legend(loc="upper right")
ax.set_title("Unit balls of three $\\ell_p$ norms")
plt.show()

## Products


`products`


In [ ]:
A = np.random.randn(3, 4)
B = np.random.randn(4, 2)

# Equation (B.10) written out literally, then the same thing vectorized.
C_loop = np.zeros((3, 2))
for i in range(3):
    for j in range(2):
        C_loop[i, j] = sum(A[i, d] * B[d, j] for d in range(4))

print("(B.10) as an explicit loop matches the @ operator:",
      np.allclose(C_loop, A @ B))

a = np.array([2.0, 1.0, 0.0])
b = np.array([1.0, 3.0, 1.0])
dot = a @ b
cos_theta = dot / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"\na . b            = {dot:.4f}")
print(f"||a|| ||b|| cos t = {np.linalg.norm(a) * np.linalg.norm(b) * cos_theta:.4f}")
print(f"angle between them = {np.degrees(np.arccos(cos_theta)):.2f} degrees")

## Inverse


`inverse`


In [ ]:
M = np.random.randn(4, 4)
N = np.random.randn(4, 4)
print("(MN)^-1 == N^-1 M^-1 :", np.allclose(np.linalg.inv(M @ N),
                                            np.linalg.inv(N) @ np.linalg.inv(M)))

# A singular matrix: the third column is the sum of the first two.
S = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 9.0],
              [7.0, 8.0, 15.0]])
print("det(S) =", round(float(np.linalg.det(S)), 12), " -> singular, no inverse")

print("\nCost of a D x D inverse, in units of elementary operations:")
print(f"{'D':>7} {'D^3':>18} {'relative to D=10':>20}")
for D in [10, 100, 1_000, 10_000]:
    print(f"{D:>7} {D**3:>18,} {D**3 / 10**3:>20,.0f}x")

## Subspaces


`subspaces`


In [ ]:
tall = np.array([[1.0, 0.0],          # 3x2: two columns, three-dimensional output
                 [0.0, 1.0],
                 [1.0, 1.0]])
wide = np.array([[1.0, 0.0, 1.0],     # 2x3: three columns, two-dimensional output
                 [0.0, 1.0, 1.0]])

for name, A in [("tall (3x2)", tall), ("wide (2x3)", wide)]:
    D1, D2 = A.shape
    rank = np.linalg.matrix_rank(A)
    print(f"{name}: rank {rank}, column space is {rank}-D inside R^{D1}, "
          f"nullspace is {D2 - rank}-D inside R^{D2}")

# The tall matrix maps all of R^2 onto a single plane in R^3: every output
# satisfies the same linear constraint, here x1 + x2 - x3 = 0.
pts = tall @ np.random.randn(2, 5)
print("\nresiduals of x1 + x2 - x3 for five random outputs:",
      np.round(pts[0] + pts[1] - pts[2], 12))

# The wide matrix reaches everything in R^2, but it has a direction it destroys.
null_dir = np.array([1.0, 1.0, -1.0])
print("wide @ nullspace direction =", wide @ null_dir)

## Eigenspectrum, determinant, and trace


`fig-ellipse`


In [ ]:
A = np.array([[2.0, -1.0],
              [3.1, 1.5]])

theta = np.linspace(0, 2 * np.pi, 721)
circle = np.stack([np.cos(theta), np.sin(theta)])
ellipse = A @ circle

U, S, Vt = np.linalg.svd(A)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.4))
axes[0].plot(circle[0], circle[1], color=NDSU_GREEN, lw=2)
axes[0].set_title(r"unit circle $\{\mathbf{x}_i\}$")

axes[1].plot(ellipse[0], ellipse[1], color=ACCENT, lw=2)
for k in range(2):
    axis = U[:, k] * S[k]
    axes[1].annotate("", xy=axis, xytext=(0, 0),
                     arrowprops=dict(arrowstyle="->", color="0.35", lw=1.6))
    axes[1].annotate(f"$\\sigma_{k+1} = {S[k]:.2f}$", xy=axis * 0.55,
                     textcoords="offset points", xytext=(6, 6), color="0.25")
axes[1].set_title(r"image $\{\mathbf{x}'_i\} = \mathbf{A}\mathbf{x}_i$")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout()
plt.show()

radii = np.linalg.norm(ellipse, axis=0)
eigvals = np.linalg.eigvals(A)
print(f"longest / shortest radius measured on the ellipse : {radii.max():.4f} / {radii.min():.4f}")
print(f"singular values of A                              : {S[0]:.4f} / {S[1]:.4f}")
print(f"eigenvalues of A                                  : {eigvals[0]:.4f}, {eigvals[1]:.4f}")
print(f"|det A| = {abs(np.linalg.det(A)):.4f}   product of singular values = {S.prod():.4f}"
      f"   product of |eigenvalues| = {np.abs(eigvals).prod():.4f}")
print(f"trace A = {np.trace(A):.4f}   sum of eigenvalues = {eigvals.sum().real:.4f}")

## Special Types of Matrix


`special-matrices`


In [ ]:
D = 5
d = np.random.uniform(1.0, 3.0, D)
Diag = np.diag(d)
print("diagonal : inverse is the reciprocal :",
      np.allclose(np.linalg.inv(Diag), np.diag(1 / d)),
      "| det == product of diagonal :", np.isclose(np.linalg.det(Diag), d.prod()))

Lower = np.tril(np.random.randn(D, D) + 3 * np.eye(D))
print("triangular: det == product of diagonal :",
      np.isclose(np.linalg.det(Lower), np.diag(Lower).prod()))

# A QR factorization is a convenient source of an orthogonal matrix.
Q, _ = np.linalg.qr(np.random.randn(D, D))
print("orthogonal: A^-1 == A^T :", np.allclose(np.linalg.inv(Q), Q.T),
      "| |eigenvalues| :", np.round(np.abs(np.linalg.eigvals(Q)), 6),
      "| det :", round(float(np.linalg.det(Q)), 6))

P = np.array([[0.0, 1.0, 0.0],
              [0.0, 0.0, 1.0],
              [1.0, 0.0, 0.0]])
abc = np.array(["a", "b", "c"])
print("\npermutation (B.16):", (P @ np.array([1.0, 2.0, 3.0])),
      "->", abc[np.argmax(P, axis=1)])
print("permutation: P^-1 == P^T :", np.allclose(np.linalg.inv(P), P.T))

## Linear Equations in Matrix Form


`linear-matrix-form`


In [ ]:
Phi0 = np.array([0.5, -1.0, 2.0])                 # phi_10, phi_20, phi_30
Phi  = np.array([[1.0, -2.0,  0.5],
                 [0.0,  1.5, -1.0],
                 [2.0,  0.0,  0.25]])
z = np.array([1.0, 2.0, -1.0])

# Equation (B.18), one scalar equation at a time.
y_scalar = np.array([Phi0[i] + sum(Phi[i, j] * z[j] for j in range(3)) for i in range(3)])

# Equation (B.19), all at once.
y_matrix = Phi0 + Phi @ z

print("scalar form :", y_scalar)
print("matrix form :", y_matrix)
print("identical   :", np.allclose(y_scalar, y_matrix))

# The same equation, as a PyTorch layer.
layer = torch.nn.Linear(3, 3)
with torch.no_grad():
    layer.weight.copy_(torch.tensor(Phi, dtype=torch.float32))
    layer.bias.copy_(torch.tensor(Phi0, dtype=torch.float32))

y_torch = layer(torch.tensor(z, dtype=torch.float32))
print("nn.Linear   :", y_torch.detach().numpy())

`batched-linear`


In [ ]:
Zbatch = np.random.randn(3, 256)                   # 256 inputs, one per column
Ybatch = Phi0[:, None] + Phi @ Zbatch              # broadcasting adds phi_0 to every column
print("input batch :", Zbatch.shape, " output batch:", Ybatch.shape)
print("column 7 matches the single-input computation:",
      np.allclose(Ybatch[:, 7], Phi0 + Phi @ Zbatch[:, 7]))

## Matrix Calculus


`jacobians`


In [ ]:
A_t = torch.tensor([[2.0, -1.0, 0.5],
                    [3.1,  1.5, -2.0]])              # 2 x 3, so D_y = 2, D_x = 3
x0 = torch.tensor([0.4, -1.2, 2.0])

J = torch.autograd.functional.jacobian(lambda v: A_t @ v, x0)
print("torch jacobian shape (numerator layout, D_y x D_x):", tuple(J.shape))
print("equals A            :", torch.allclose(J, A_t))
print("its transpose is Prince's dy/dx = A^T, shape", tuple(J.T.shape))

# Scalar-valued function of a vector: the derivative is a vector of the same size.
x = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()                                    # y = ||x||^2, so dy/dx = 2x
y.backward()
print("\nd/dx of ||x||^2 :", x.grad.numpy(), " expected 2x :", (2 * x).detach().numpy())

`autograd-vs-analytic`


In [ ]:
X = torch.randn(20, 3)
y_obs = torch.randn(20)
phi = torch.zeros(3, requires_grad=True)

loss = ((X @ phi - y_obs) ** 2).sum()
loss.backward()

analytic = 2 * X.T @ (X @ phi.detach() - y_obs)      # dL/dphi = 2 X^T (X phi - y)
print("autograd :", phi.grad.numpy().round(4))
print("analytic :", analytic.numpy().round(4))
print("agree    :", torch.allclose(phi.grad, analytic, atol=1e-5))

## The same loss, in matrix form


`lr-data`


In [ ]:
I = 24
temp = np.random.uniform(140, 200, I)                      # cure temperature, deg C
strength = 12.0 + 0.42 * temp + np.random.normal(0, 2.5, I)  # tensile strength, MPa

X = np.column_stack([np.ones(I), temp])                    # design matrix, I x 2
print("design matrix:", X.shape, " first two rows:\n", X[:2])
print("\nnormal equations X^T X phi = X^T y:")
print("X^T X =\n", np.round(X.T @ X, 2))

## Training, three ways


`lr-three-ways`


In [ ]:
# 1. Closed form: solve the normal equations. Note that we *solve*, never invert.
phi_closed = np.linalg.solve(X.T @ X, X.T @ strength)

# 2. Gradient descent, using equation (B.21) via the gradient in @eq-lr-gradient.
def gradient_descent(X, y, lr, n_steps):
    phi = np.zeros(X.shape[1])
    history = []
    for _ in range(n_steps):
        phi = phi - lr * (2 * X.T @ (X @ phi - y))
        history.append(np.sum((X @ phi - y) ** 2))
    return phi, np.array(history)

phi_gd, _ = gradient_descent(X, strength, lr=1e-6, n_steps=20_000)

# 3. PyTorch: the same model as one nn.Linear layer, trained by autograd.
Xt = torch.tensor(temp, dtype=torch.float32).unsqueeze(1)
Yt = torch.tensor(strength, dtype=torch.float32).unsqueeze(1)

model = torch.nn.Linear(1, 1)                     # exactly equation (2.4)
opt = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()
for _ in range(4_000):
    opt.zero_grad()
    loss_fn(model(Xt), Yt).backward()
    opt.step()

print(f"{'method':<22}{'phi_0':>10}{'phi_1':>10}{'loss':>12}")
for name, p in [("closed form", phi_closed),
                ("gradient descent", phi_gd),
                ("PyTorch (Adam)", np.array([model.bias.item(), model.weight.item()]))]:
    print(f"{name:<22}{p[0]:>10.4f}{p[1]:>10.4f}{np.sum((X @ p - strength) ** 2):>12.4f}")

## Why the iterative methods struggle


`fig-conditioning`


In [ ]:
z = (temp - temp.mean()) / temp.std()
Xz = np.column_stack([np.ones(I), z])

for name, M in [("raw", X), ("standardized", Xz)]:
    ev = np.linalg.eigvalsh(2 * M.T @ M)
    print(f"{name:<14} Hessian eigenvalues {ev[0]:12.4f} {ev[1]:14.4f}"
          f"   condition number {ev[1] / ev[0]:12.1f}")

ev_raw = np.linalg.eigvalsh(2 * X.T @ X)
ev_std = np.linalg.eigvalsh(2 * Xz.T @ Xz)
lr_raw, lr_std = 0.9 * 2 / ev_raw[1], 0.9 * 2 / ev_std[1]

n_steps = 2_000
phi_raw, hist_raw = gradient_descent(X,  strength, lr=lr_raw, n_steps=n_steps)
phi_std, hist_std = gradient_descent(Xz, strength, lr=lr_std, n_steps=n_steps)

best = np.sum((X @ phi_closed - strength) ** 2)
reached = [int(np.argmax(h <= best * 1.01)) + 1 if (h <= best * 1.01).any() else None
           for h in (hist_raw, hist_std)]
print(f"\nlargest stable step size -- raw: {lr_raw:.3e}, standardized: {lr_std:.3e}")
print(f"steps to get within 1% of the optimum -- raw: {reached[0]}, standardized: {reached[1]}")

def descent_path(M, lr, n=400):
    """The first n gradient descent steps, starting from the origin."""
    path = [np.zeros(2)]
    for _ in range(n):
        path.append(path[-1] - lr * (2 * M.T @ (M @ path[-1] - strength)))
    return np.array(path)

fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.8))

for ax, (title, M, lr) in zip(axes[:2], [("raw input", X, lr_raw),
                                         ("standardized input", Xz, lr_std)]):
    phi_hat = np.linalg.solve(M.T @ M, M.T @ strength)
    path = descent_path(M, lr)

    # A square, equal-aspect window centred on the path, so that a condition
    # number of one really does look like a circle.
    corners = np.vstack([path, phi_hat])
    centre = 0.5 * (corners.min(axis=0) + corners.max(axis=0))
    half = 0.7 * (corners.max(axis=0) - corners.min(axis=0)).max() + 0.5
    lo, hi = centre - half, centre + half

    g0, g1 = np.meshgrid(np.linspace(lo[0], hi[0], 250),
                         np.linspace(lo[1], hi[1], 250))
    Lg = ((g0[..., None] + g1[..., None] * M[:, 1] - strength) ** 2).sum(axis=-1)
    ax.contourf(g0, g1, np.log10(Lg), levels=25, cmap="magma")
    ax.contour(g0, g1, np.log10(Lg), levels=12, colors="white", linewidths=0.4, alpha=0.5)

    ax.plot(path[:, 0], path[:, 1], color=NDSU_GOLD, lw=1.6)
    ax.plot(path[::20, 0], path[::20, 1], "o", color=NDSU_GOLD, ms=3)
    ax.plot(*phi_hat, "*", color="white", ms=16, mec="k", mew=0.7)
    ax.set_title(f"{title}: $\\kappa = {np.linalg.cond(M.T @ M):.3g}$")
    ax.set_xlabel("$\\phi_0$"); ax.set_ylabel("$\\phi_1$")
    ax.set_aspect("equal")
    ax.grid(False)

steps = np.arange(1, n_steps + 1)
axes[2].plot(steps, hist_raw, color=ACCENT, label="raw")
axes[2].plot(steps, hist_std, color=NDSU_GREEN, label="standardized")
axes[2].axhline(best, color="0.4", ls="--", lw=1, label="closed form")
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_xlabel("gradient descent step"); axes[2].set_ylabel("$L[\\boldsymbol{\\phi}]$")
axes[2].set_title("convergence")
axes[2].legend()
plt.tight_layout()
plt.show()

## Testing


`fig-under-overfit`


In [ ]:
# A fresh batch of parts off the same process, as an honest test set.
temp_test = np.random.uniform(140, 200, 200)
strength_test = 12.0 + 0.42 * temp_test + np.random.normal(0, 2.5, 200)

# Standardize using the *training* statistics only -- the test set must never
# leak into the fit, and the conditioning section above says why scaling matters.
mu, sigma = temp.mean(), temp.std()
z_train = (temp - mu) / sigma
z_test = (temp_test - mu) / sigma
z_grid = (np.linspace(138, 202, 400) - mu) / sigma

fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.8), sharey=True)
for ax, (degree, label) in zip(axes, [(0, "underfitting"),
                                      (1, "the right model"),
                                      (12, "overfitting")]):
    coef = np.polyfit(z_train, strength, degree)
    train_loss = np.mean((np.polyval(coef, z_train) - strength) ** 2)
    test_loss = np.mean((np.polyval(coef, z_test) - strength_test) ** 2)

    ax.scatter(temp_test, strength_test, s=10, facecolors="none",
               edgecolors="0.65", linewidths=0.6, zorder=2)
    ax.scatter(temp, strength, s=34, color=ACCENT, zorder=3)
    ax.plot(np.linspace(138, 202, 400), np.polyval(coef, z_grid),
            color=NDSU_GREEN, lw=2, zorder=4)
    ax.set_title(f"degree {degree}: {label}\ntrain {train_loss:.2f}   test {test_loss:,.2f}")
    ax.set_xlabel("cure temperature (°C)")

axes[0].set_ylabel("tensile strength (MPa)")
axes[0].set_ylim(strength.min() - 12, strength.max() + 12)
plt.tight_layout()
plt.show()